In [1]:
import kagglehub
import pandas as pd
from pathlib import Path

path= Path(kagglehub.dataset_download("gpreda/reddit-wallstreetsbets-posts"))
print(list(path.rglob("*.csv")))

100%|██████████| 16.5M/16.5M [00:03<00:00, 5.17MB/s]

Extracting files...


[WindowsPath('C:/Users/Dell/.cache/kagglehub/datasets/gpreda/reddit-wallstreetsbets-posts/versions/106/reddit_wsb.csv')]


In [2]:
reddit = pd.read_csv(path / "reddit_wsb.csv")
print(reddit.shape)
print(reddit.columns.tolist())
reddit.head()

(53187, 8)
['title', 'score', 'id', 'url', 'comms_num', 'created', 'body', 'timestamp']


,title,score,id,url,comms_num,created,body,timestamp
0,"It's not about the money, it's about sending a...",55,l6ulcx,https://v.redd.it/6j75regs72e61,6,1.611863e+09,NaN,2021-01-28 21:37:41
1,Math Professor Scott Steiner says the numbers ...,110,l6uibd,https://v.redd.it/ah50lyny62e61,23,1.611862e+09,NaN,2021-01-28 21:32:10
2,Exit the system,0,l6uhhn,https://www.reddit.com/r/wallstreetbets/commen...,47,1.611862e+09,The CEO of NASDAQ pushed to halt trading “to g...,2021-01-28 21:30:35
3,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,29,l6ugk6,https://sec.report/Document/0001193125-21-019848/,74,1.611862e+09,NaN,2021-01-28 21:28:57
4,"Not to distract from GME, just thought our AMC...",71,l6ufgy,https://i.redd.it/4h2sukb662e61.jpg,156,1.611862e+09,NaN,2021-01-28 21:26:56


In [3]:
import re
COMPANY_NAMES = {
    "AAPL": ["Apple"],
    "MSFT": ["Microsoft"],
    "GOOGL": ["Google", "Alphabet", "GOOG"],
    "AMZN": ["Amazon"],
    "NVDA": ["Nvidia"],
    "TSLA": ["Tesla"],
    "META": ["Facebook", "Meta Platforms"],
    "JPM": ["JPMorgan", "JP Morgan"],
    "JNJ": ["Johnson & Johnson"],
    "XOM": ["Exxon", "ExxonMobil"],
}
def find_tickers(text:str)->list[str]:
    found=[]
    for ticker, names in COMPANY_NAMES.items():
        symbol_hit = re.search(rf"\${ticker}\b|\b{ticker}\b", text)   # case-SENSITIVE
        name_pattern = "|".join(rf"\b{re.escape(n)}\b" for n in names)
        name_hit = re.search(name_pattern, text, re.IGNORECASE)
        if symbol_hit or name_hit:
            found.append(ticker)
    return found


In [4]:
tests = [
    "just YOLO'd into tsla 🚀",              # lowercase symbol → only "Tesla" name would hit... no name here → []
    "TSLA and NVDA printing today",           # → ['NVDA', 'TSLA']
    "$AAPL calls",                            # → ['AAPL']
    "the meta of this sub is insane",         # → [] (the META trap, dodged)
    "Apple pie recipe",                       # → ['AAPL'] — known false positive, document it
    "Buying Johnson & Johnson for the divvy", # → ['JNJ']
]
for t in tests:
    print(find_tickers(t), "|", t)

[] | just YOLO'd into tsla 🚀
['NVDA', 'TSLA'] | TSLA and NVDA printing today
['AAPL'] | $AAPL calls
[] | the meta of this sub is insane
['AAPL'] | Apple pie recipe
['JNJ'] | Buying Johnson & Johnson for the divvy


# Part 2 — Transcript dataset hunt
Probing two candidate sources; whichever shows real files + coverage wins.

In [5]:
from huggingface_hub import list_repo_files

print(list_repo_files("jlh-ibm/earnings_call", repo_type="dataset"))

['.gitattributes', 'README.md', 'data/stock_prices/AAPL.csv', 'data/stock_prices/AMD.csv', 'data/stock_prices/AMZN.csv', 'data/stock_prices/ASML.csv', 'data/stock_prices/CSCO.csv', 'data/stock_prices/GOOGL.csv', 'data/stock_prices/INTC.csv', 'data/stock_prices/MSFT.csv', 'data/stock_prices/MU.csv', 'data/stock_prices/NASDAQ.csv', 'data/stock_prices/NVDA.csv', 'data/transcripts/AAPL/2016-Apr-26-AAPL.txt', 'data/transcripts/AAPL/2016-Jan-26-AAPL.txt', 'data/transcripts/AAPL/2016-Jul-26-AAPL.txt', 'data/transcripts/AAPL/2016-Oct-25-AAPL.txt', 'data/transcripts/AAPL/2017-Aug-01-AAPL.txt', 'data/transcripts/AAPL/2017-Jan-31-AAPL.txt', 'data/transcripts/AAPL/2017-May-02-AAPL.txt', 'data/transcripts/AAPL/2017-Nov-02-AAPL.txt', 'data/transcripts/AAPL/2018-Feb-01-AAPL.txt', 'data/transcripts/AAPL/2018-Jul-31-AAPL.txt', 'data/transcripts/AAPL/2018-May-01-AAPL.txt', 'data/transcripts/AAPL/2018-Nov-01-AAPL.txt', 'data/transcripts/AAPL/2019-Apr-30-AAPL.txt', 'data/transcripts/AAPL/2019-Jan-29-AAPL.

In [7]:
import kagglehub
from pathlib import Path

p = Path(kagglehub.dataset_download("tpotterer/motley-fool-scraped-earnings-call-transcripts"))
print(list(p.rglob("*"))[:15])

100%|██████████| 269M/269M [00:33<00:00, 8.39MB/s] 

Extracting files...


[WindowsPath('C:/Users/Dell/.cache/kagglehub/datasets/tpotterer/motley-fool-scraped-earnings-call-transcripts/versions/1/motley-fool-data.pkl')]
